In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("statcast_2026.parquet")

# The columns we need for trajectory math:
# release_pos_x/y/z -- where the ball starts
# vx0/vy0/vz0 -- initial velocity in each direction
# ax/ay/az -- acceleration in each direction (gravity + spin-induced movement)
trajectory_cols = [
    'release_pos_x', 'release_pos_y', 'release_pos_z',
    'vx0', 'vy0', 'vz0', 'ax', 'ay', 'az'
]

# Drop any pitches missing trajectory data -- can't do the physics without it
traj_df = df.dropna(subset=trajectory_cols).copy()
print(f"Pitches with complete trajectory data: {len(traj_df):,} of {len(df):,} "
      f"({len(traj_df)/len(df):.1%})")

# Average trajectory parameters per (pitcher, pitch_type) -- same
# granularity we've used since Phase 3, treating each pitcher's pitch
# type as having one "typical" trajectory rather than pitch-by-pitch
avg_trajectory = traj_df.groupby(['pitcher', 'pitch_type'])[trajectory_cols].mean().reset_index()

# Sanity check: vy0 should be consistently NEGATIVE for every pitch --
# in Statcast's coordinate system, y decreases as the ball travels
# toward the plate, so a pitch moving toward the batter has negative
# vy0. A positive value here would mean something is wrong.
print(f"\nPitches with positive vy0 (would indicate a data issue): "
      f"{(avg_trajectory['vy0'] > 0).sum()} of {len(avg_trajectory)}")

avg_trajectory.head()

Pitches with complete trajectory data: 556,412 of 558,489 (99.6%)

Pitches with positive vy0 (would indicate a data issue): 0 of 3745


,pitcher,pitch_type,release_pos_x,release_pos_y,release_pos_z,vx0,vy0,vz0,ax,ay,az
0,434378,CH,-1.78125,54.5125,7.02,5.516539,-121.414916,-6.433716,-11.951936,23.784103,-25.118983
1,434378,CU,-1.667143,54.617143,7.077143,2.593261,-115.294087,-3.544165,2.757319,21.673023,-39.76293
2,434378,FF,-1.547059,54.609412,7.089706,6.856275,-135.219481,-9.907893,-10.972221,30.246027,-11.700741
3,434378,SL,-1.572727,54.603636,7.119091,5.139239,-127.079218,-7.801646,0.229715,25.766645,-25.888068
4,434378,ST,-1.615,54.66,7.01,3.801786,-119.493515,-5.504445,7.853421,25.077706,-32.889431


In [2]:
# Solve for time-to-commit-point using the y-axis (depth) trajectory.
# Kinematics: y(t) = release_pos_y + vy0*t + 0.5*ay*t^2
# We want the time t where y(t) = 25 (our commit point).
# This is a quadratic equation in t, solved with the quadratic formula.

COMMIT_Y = 25.0  # feet from home plate, per standard tunneling convention

def solve_time_to_y(row, target_y):
    # Rearranged to standard quadratic form: 0.5*ay*t^2 + vy0*t + (release_pos_y - target_y) = 0
    a = 0.5 * row['ay']
    b = row['vy0']
    c = row['release_pos_y'] - target_y

    discriminant = b**2 - 4*a*c
    if discriminant < 0:
        return np.nan  # shouldn't happen with real pitch data, but guards against bad rows

    # Two possible solutions from the quadratic formula -- we want the
    # smaller positive one (the first time the ball passes through that y)
    t1 = (-b + np.sqrt(discriminant)) / (2*a)
    t2 = (-b - np.sqrt(discriminant)) / (2*a)
    valid_times = [t for t in [t1, t2] if t > 0]
    return min(valid_times) if valid_times else np.nan

avg_trajectory['time_to_commit'] = avg_trajectory.apply(lambda r: solve_time_to_y(r, COMMIT_Y), axis=1)

# Sanity check: time to commit point should be a small, positive number
# of seconds -- roughly 0.2-0.3s for most pitches, since release-to-plate
# is only about 0.4s total for a fastball
print(avg_trajectory['time_to_commit'].describe())

count    3745.000000
mean        0.238266
std         0.038669
min         0.195350
25%         0.219833
50%         0.231977
75%         0.244726
max         0.689098
Name: time_to_commit, dtype: float64


In [3]:
# Investigate the outlier(s) in time_to_commit -- sort descending and
# look at the pitchers/pitch types with the longest calculated times
outliers = avg_trajectory.sort_values('time_to_commit', ascending=False).head(10)
print(outliers[['pitcher', 'pitch_type', 'time_to_commit', 'ay', 'vy0', 'release_pos_y']])

      pitcher pitch_type  time_to_commit        ay        vy0  release_pos_y
1338   664059         EP        0.689098  4.661721 -48.133211      57.061667
2842   686894         CU        0.658435  4.621469 -48.769885          56.11
1017   656716         EP        0.622416    5.0181 -52.974931      57.000417
875    644433         EP        0.610580  4.354782 -54.053321      57.192143
1136   660844         EP        0.595021  5.520403 -55.881174      57.273226
101    543510         EP        0.590287  5.278298 -54.165841      56.053818
1373   664238         EP        0.589229  5.268513 -54.921372      56.446667
684    624431         EP        0.569941  5.113274 -56.959682      56.633171
302    596142         CU        0.566252  5.805948 -57.681192       56.73125
2647   682668         EP        0.563304  5.344512 -56.706853      56.095238


In [4]:
# Now that we know WHEN each pitch reaches the commit point (y=25),
# plug that time into the x and z position formulas to find WHERE the
# ball actually is at that moment.
# Kinematics: position(t) = start + velocity*t + 0.5*acceleration*t^2

t = avg_trajectory['time_to_commit']

avg_trajectory['commit_x'] = (
    avg_trajectory['release_pos_x'] + avg_trajectory['vx0'] * t + 0.5 * avg_trajectory['ax'] * t**2
)
avg_trajectory['commit_z'] = (
    avg_trajectory['release_pos_z'] + avg_trajectory['vz0'] * t + 0.5 * avg_trajectory['az'] * t**2
)

# Sanity check: commit_x and commit_z should be somewhere in a
# reasonable "in-flight" range -- not still at the release point, but
# not yet all the way to the ball's final plate position either, since
# the commit point is partway through the pitch's flight
print(avg_trajectory[['commit_x', 'commit_z']].describe())
avg_trajectory[['pitcher', 'pitch_type', 'commit_x', 'commit_z']].head()

       commit_x  commit_z
count    3745.0    3745.0
mean  -0.319811   4.27996
std    0.849787  0.426945
min   -3.070706  2.370667
25%   -0.915243  4.053135
50%   -0.595595   4.25405
75%    0.376882  4.443564
max    3.251262  9.512566


,pitcher,pitch_type,commit_x,commit_z
0,434378,CH,-0.777763,4.63738
1,434378,CU,-0.888412,4.764177
2,434378,FF,-0.283819,4.569075
3,434378,SL,-0.33928,4.518867
4,434378,ST,-0.38998,4.536497


In [5]:
# For every pitcher, build every possible PAIR of their pitch types
# (not triplets yet -- we'll combine pairs into full 3-pitch combos after),
# and calculate the tunneling distance between them at the commit point.
from itertools import combinations

tunnel_rows = []
for pitcher_id, group in avg_trajectory.groupby('pitcher'):
    pitches = group.set_index('pitch_type')[['commit_x', 'commit_z']]
    for pt1, pt2 in combinations(pitches.index, 2):
        # Euclidean distance between the two pitches' positions at the
        # commit point -- straightforward "how far apart do they look"
        dist = np.sqrt(
            (pitches.loc[pt1, 'commit_x'] - pitches.loc[pt2, 'commit_x'])**2 +
            (pitches.loc[pt1, 'commit_z'] - pitches.loc[pt2, 'commit_z'])**2
        )
        tunnel_rows.append({
            'pitcher': pitcher_id,
            'pitch_a': pt1,
            'pitch_b': pt2,
            'tunnel_distance': dist
        })

tunnel_pairs = pd.DataFrame(tunnel_rows)
print(f"Total pitch-pair rows: {len(tunnel_pairs):,}")
print(tunnel_pairs['tunnel_distance'].describe())
tunnel_pairs.head()

Total pitch-pair rows: 7,532
count    7532.000000
mean        0.419212
std         0.392332
min         0.002810
25%         0.207986
50%         0.330153
75%         0.491266
max         6.014254
Name: tunnel_distance, dtype: float64


,pitcher,pitch_a,pitch_b,tunnel_distance
0,434378,CH,CU,0.168287
1,434378,CH,FF,0.498645
2,434378,CH,SL,0.454217
3,434378,CH,ST,0.400691
4,434378,CU,FF,0.635293


In [6]:
# Build a fast lookup: (pitcher, pitch_a, pitch_b) -> distance.
# Since pairs are unordered, we store both directions so a lookup works
# regardless of which pitch comes first when we query it later.
tunnel_lookup = {}
for row in tunnel_pairs.itertuples():
    tunnel_lookup[(row.pitcher, row.pitch_a, row.pitch_b)] = row.tunnel_distance
    tunnel_lookup[(row.pitcher, row.pitch_b, row.pitch_a)] = row.tunnel_distance

# Load the combo list with run value scores from Phase 3
all_combos = pd.read_parquet("combos_handedness_only.parquet")

def combo_tunnel_score(row):
    p = row['pitcher']
    pitches = [row['pitch_1'], row['pitch_2'], row['pitch_3']]
    # Average the 3 pairwise distances within this combo (pitch1-pitch2,
    # pitch1-pitch3, pitch2-pitch3) into one overall tunneling score --
    # lower = the 3 pitches look more similar to each other at the
    # commit point, i.e., better tunneled as a group
    dists = []
    for pt1, pt2 in combinations(pitches, 2):
        key = (p, pt1, pt2)
        if key in tunnel_lookup:
            dists.append(tunnel_lookup[key])
    return np.mean(dists) if dists else np.nan

all_combos['tunnel_score'] = all_combos.apply(combo_tunnel_score, axis=1)

# Average the L and R run value scores into one overall score per combo,
# so we have a single number to correlate against tunnel_score
all_combos['avg_run_value'] = all_combos[['combo_score_vs_L_weighted', 'combo_score_vs_R_weighted']].mean(axis=1)

# The actual test: does better tunneling (LOWER distance) correlate with
# better run value (LOWER/more negative score)? A POSITIVE correlation
# would confirm this -- worse tunneling (higher distance) going with
# worse run value (higher/less negative score) moving together.
correlation = all_combos[['tunnel_score', 'avg_run_value']].corr().iloc[0, 1]
print(f"Correlation between tunnel_score and run value: {correlation:.3f}")

Correlation between tunnel_score and run value: -0.072


In [7]:
# Solve for time to reach the front of home plate (y=1.417), same
# quadratic approach as before, just a different target_y
PLATE_Y = 1.417

avg_trajectory['time_to_plate'] = avg_trajectory.apply(lambda r: solve_time_to_y(r, PLATE_Y), axis=1)

# Remaining time AFTER the commit point -- this is the timing budget
# a batter has to react once the pitches start diverging. If two
# pitches have very different remaining times, one arrives noticeably
# sooner/later, giving the batter an extra clue even if they looked
# spatially similar at the commit point.
avg_trajectory['remaining_time'] = avg_trajectory['time_to_plate'] - avg_trajectory['time_to_commit']

print(avg_trajectory['remaining_time'].describe())

count    3745.000000
mean        0.201787
std         0.029370
min         0.171431
25%         0.186795
50%         0.196940
75%         0.207912
max         0.540128
Name: remaining_time, dtype: float64


In [8]:
# Build pairwise REMAINING TIME differences, same approach as the
# spatial distance calculation before
timing_rows = []
for pitcher_id, group in avg_trajectory.groupby('pitcher'):
    times = group.set_index('pitch_type')['remaining_time']
    for pt1, pt2 in combinations(times.index, 2):
        # Absolute difference in remaining flight time -- how much of a
        # timing mismatch exists between these two pitches after the
        # commit point
        time_diff = abs(times[pt1] - times[pt2])
        timing_rows.append({
            'pitcher': pitcher_id,
            'pitch_a': pt1,
            'pitch_b': pt2,
            'timing_diff': time_diff
        })

timing_pairs = pd.DataFrame(timing_rows)
print(timing_pairs['timing_diff'].describe())

# Merge timing differences onto the existing spatial distance table
tunnel_pairs = tunnel_pairs.merge(timing_pairs, on=['pitcher', 'pitch_a', 'pitch_b'])

# Build a composite score: since tunnel_distance is in feet (~0-1 range)
# and timing_diff is in seconds (~0-0.1 range), we need to put them on
# comparable scales before combining -- z-score standardize both, same
# approach we used back in Phase 2 for clustering features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
tunnel_pairs[['dist_z', 'timing_z']] = scaler.fit_transform(tunnel_pairs[['tunnel_distance', 'timing_diff']])

# Composite tunneling score = average of the two standardized measures.
# Lower = better tunneled on BOTH dimensions (close together in space
# AND arriving at similar times)
tunnel_pairs['composite_tunnel'] = tunnel_pairs[['dist_z', 'timing_z']].mean(axis=1)

tunnel_pairs.head()

count    7532.000000
mean        0.017393
std         0.017892
min         0.000003
25%         0.007771
50%         0.014852
75%         0.022893
max         0.306702
Name: timing_diff, dtype: float64


,pitcher,pitch_a,pitch_b,tunnel_distance,timing_diff,dist_z,timing_z,composite_tunnel
0,434378,CH,CU,0.168287,0.011302,-0.639617,-0.340434,-0.490025
1,434378,CH,FF,0.498645,0.020902,0.202475,0.196133,0.199304
2,434378,CH,SL,0.454217,0.009439,0.089227,-0.444582,-0.177677
3,434378,CH,ST,0.400691,0.004891,-0.047212,-0.698825,-0.373018
4,434378,CU,FF,0.635293,0.032204,0.550797,0.827884,0.689340


In [9]:
# Rebuild the fast lookup using composite_tunnel instead of just distance
composite_lookup = {}
for row in tunnel_pairs.itertuples():
    composite_lookup[(row.pitcher, row.pitch_a, row.pitch_b)] = row.composite_tunnel
    composite_lookup[(row.pitcher, row.pitch_b, row.pitch_a)] = row.composite_tunnel

def combo_composite_tunnel_score(row):
    p = row['pitcher']
    pitches = [row['pitch_1'], row['pitch_2'], row['pitch_3']]
    scores = []
    for pt1, pt2 in combinations(pitches, 2):
        key = (p, pt1, pt2)
        if key in composite_lookup:
            scores.append(composite_lookup[key])
    return np.mean(scores) if scores else np.nan

all_combos['composite_tunnel_score'] = all_combos.apply(combo_composite_tunnel_score, axis=1)

# Re-test: does the composite (space + timing) tunneling score correlate
# with run value better than spatial distance alone did?
correlation_composite = all_combos[['composite_tunnel_score', 'avg_run_value']].corr().iloc[0, 1]
print(f"Correlation with SPATIAL distance alone: -0.072")
print(f"Correlation with COMPOSITE (space + timing): {correlation_composite:.3f}")

Correlation with SPATIAL distance alone: -0.072
Correlation with COMPOSITE (space + timing): -0.030


In [10]:
# Save the tunneling analysis for the write-up -- both the pairwise
# tunneling table and the combo-level scores with their correlation
# to run value, so the numbers behind this finding are on record
tunnel_pairs.to_parquet("phase7_tunnel_pairs.parquet", index=False)

all_combos[['pitcher', 'pitch_1', 'pitch_2', 'pitch_3', 'tunnel_score',
            'composite_tunnel_score', 'avg_run_value']].to_csv(
    "phase7_tunneling_vs_run_value.csv", index=False
)

print("Saved phase7_tunnel_pairs.parquet and phase7_tunneling_vs_run_value.csv")

Saved phase7_tunnel_pairs.parquet and phase7_tunneling_vs_run_value.csv
